In [1]:
import sys
import pathlib
import numpy as np
import pandas as pd

PROJECT_ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from common.config import DATASET_FILE, METADATA_FILE

df   = pd.read_csv(DATASET_FILE)
meta = pd.read_csv(METADATA_FILE)

print(f"Countries : {len(df)}")
print(f"Criteria  : {len(meta)}")
df.head()

Countries : 26
Criteria  : 8


,Country,Employment rate,Long-term unemployment rate,Personal earnings,Life expectancy,Life satisfaction,Employees working very long hours,Air pollution,Distance from Poznan (km)
0,Austria,72.0,1.3,53132.0,82.0,7.2,5.3,12.2,468.5
1,Belgium,65.0,2.3,54327.0,82.1,6.8,4.3,12.8,883.8
2,Czech Republic,74.0,0.6,29885.0,79.3,6.9,4.5,17.0,311.7
3,Denmark,74.0,0.9,58430.0,81.5,7.5,1.1,10.0,461.5
4,Estonia,74.0,1.2,30720.0,78.8,6.5,2.2,5.9,920.1


In [2]:
from ahp.hierarchy_setup import CATEGORIES, CATEGORY_CRITERIA, CRITERION_CATEGORY, CRITERIA, DIRECTIONS

print(CRITERIA)
print(DIRECTIONS)

['Employment rate', 'Long-term unemployment rate', 'Personal earnings', 'Life expectancy', 'Life satisfaction', 'Employees working very long hours', 'Air pollution', 'Distance from Poznan (km)']
{'Employment rate': 1, 'Long-term unemployment rate': -1, 'Personal earnings': 1, 'Life expectancy': 1, 'Life satisfaction': 1, 'Employees working very long hours': -1, 'Air pollution': -1, 'Distance from Poznan (km)': -1}


In [3]:
from ahp.weights import ahp_weights

A_test = np.array([
    [1,   3],
    [1/3, 1],
])

result = ahp_weights(A_test)
print(result["weights"])   # should be roughly [0.75, 0.25]
print(result["lambda_max"])
# should be exactly 2.0 for a 2x2 consistent matrix
print(f"CR = {result['CR']:.4f}")
print(f"Consistent: {result['consistent']}")

[0.75 0.25]
2.0
CR = 0.0000
Consistent: True


## AHP Hierarchy and Pairwise Comparison Matrices

The decision problem is structured as a 3-level hierarchy:

**Goal:** Find the best European country to live in for a student from Poznań.
```
GOAL
├── Economic
│       ├── Employment rate
│       ├── Long-term unemployment rate
│       └── Personal earnings
├── Social/Health
│       ├── Life expectancy
│       ├── Life satisfaction
│       └── Employees working very long hours
└── Geography/Environment
        ├── Air pollution
        └── Distance from Poznan (km)
```

At each level the DM provides pairwise comparisons using Saaty's 1-9 scale:

| Score | Meaning |
|-------|---------|
| 1 | Equal importance |
| 3 | Moderate importance |
| 5 | Strong importance |
| 7 | Very strong importance |
| 9 | Extreme importance |

**DM judgments (Goal level):**
- Economic is **strongly** more important than Social/Health (5×)
- Economic is **moderately** more important than Geography/Environment (3×)
- Social/Health is **slightly** more important than Geography/Environment (2×)

Note: the Goal matrix is intentionally inconsistent (CR=0.14 > 0.10) —
the three judgments are not perfectly transitive, which is natural in human judgment.

In [5]:
from ahp.dm_matrices import MATRICES
from ahp.weights import ahp_weights

for name, A in MATRICES.items():
    r = ahp_weights(A)
    mark = "Satisfactory" if r["consistent"] else "Inconsistent"
    print(f"{name:<28s}  CR={r['CR']:.4f}  {mark}")

Goal                          CR=0.1407  Inconsistent
Economic                      CR=0.0462  Satisfactory
Social/Health                 CR=0.0079  Satisfactory
Geography/Environment         CR=0.0000  Satisfactory
